In [3]:
import sys
print(sys.executable)

C:\Users\HP\anaconda3\python.exe


In [4]:
%pip install --upgrade chromadb

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pybase64-1.5.1-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached onnxruntime-1.31.0-cp313-cp313-win_amd64.whl.metadata (6.0 kB)
  Using cached opentelemetry_api-1.45.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.45.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached kubernetes-37.0.1-py2.py3-none-any.whl.metadata (1.9 kB)
  Using cached mmh3-5.3.1-cp313-cp313-win_amd64.whl.metadata (14 kB)
  Using cached pyproject_hooks-1.3.3-py3-none-any.whl.metadata (1.3 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.wh

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.25.0 requires botocore<1.40.50,>=1.40.46, but you have botocore 1.43.111 which is incompatible.
s3fs 2025.10.0 requires fsspec==2025.10.0, but you have fsspec 2026.9.0 which is incompatible.


In [5]:
import chromadb
print("ChromaDB version:", chromadb.__version__)

ChromaDB version: 1.5.9


In [6]:
import sys
import subprocess

print("Python executable:", sys.executable)

subprocess.run([
    sys.executable, "-m", "pip", "show", "chromadb"
])

Python executable: C:\Users\HP\anaconda3\python.exe


CompletedProcess(args=['C:\\Users\\HP\\anaconda3\\python.exe', '-m', 'pip', 'show', 'chromadb'], returncode=0)

In [9]:
from pathlib import Path

import chromadb

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# STEP 1: CREATE SAMPLE KNOWLEDGE DOCUMENT
# Replace these notes with your own lecture material if required.

knowledge_text = """
Artificial Intelligence (AI) is a field of computer science
that develops systems capable of performing intelligent tasks.

Machine Learning (ML) is a subset of AI. It enables computers
to learn patterns from data and make predictions.

Deep Learning is a subset of machine learning that uses
multi-layer neural networks.

Natural Language Processing (NLP) enables computers to
understand and generate human language.

Retrieval-Augmented Generation (RAG) combines document
retrieval with a language model to generate answers using
information retrieved from a knowledge base.

LangChain connects language models, prompts, retrievers,
document loaders and other components.

ChromaDB is a vector database that stores embeddings and
retrieves relevant text using vector similarity.

Ollama allows language models and embedding models to
run locally on a computer.
"""

Path("AI_Notes.txt").write_text(
    knowledge_text, encoding="utf-8"
)

print("STEP 1: Knowledge document created.")

# STEP 2: LOAD THE DOCUMENT

loader = TextLoader(
    "AI_Notes.txt", encoding="utf-8"
)

documents = loader.load()
print("STEP 2: Documents loaded:", len(documents))




# STEP 3: SPLIT DOCUMENT INTO CHUNKS

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("STEP 3: Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}:")
    print(chunk.page_content)

# STEP 4: CREATE EMBEDDING MODEL

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("\nSTEP 4: Embedding model configured.")

# STEP 5: CREATE CHROMADB VECTOR DATABASE
# Use a persistent local database.
# Clear this demo collection on reruns to avoid duplicate chunks.
client = chromadb.PersistentClient(
    path="./chroma_db"
)
collection_name = "undergraduate_ai_notes"
existing_collections = [
    collection.name for collection in client.list_collections()
]

if collection_name in existing_collections:
    client.delete_collection(name=collection_name)

vectorstore = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embeddings
)
vectorstore.add_documents(chunks)
print("STEP 5: Chunks stored in ChromaDB.")
# STEP 6: CREATE RETRIEVER
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("STEP 6: Retriever created.")
# STEP 7: CONFIGURE OLLAMA LANGUAGE MODEL
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)
print("STEP 7: Ollama language model configured.")
# STEP 8: CREATE PROMPT TEMPLATE
prompt = ChatPromptTemplate.from_template("""
You are an AI teaching assistant.

Answer the question using the supplied context.
If the context does not contain the answer, clearly say
that the provided notes do not contain enough information.

Context:
{context}

Question:
{question}

Answer:
""")


# STEP 9: FORMAT RETRIEVED DOCUMENTS

def format_documents(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )


# STEP 10: BUILD THE COMPLETE RAG CHAIN

rag_chain = (
    {
        "context": retriever | format_documents,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("STEP 8-10: Complete RAG chain created.")


# STEP 11: ASK A QUESTION

question = "What is the difference between AI and ML?"

# Change the question above to ask something else.

print("\n" + "=" * 60)
print("USER QUESTION:", question)
print("=" * 60)

# Display the chunks retrieved for this question.
retrieved_docs = retriever.invoke(question)

print("\nRETRIEVED CONTEXT:")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nRetrieved Chunk {i}:")
    print(doc.page_content)

# Generate the answer using retrieved context.
answer = rag_chain.invoke(question)

print("\n" + "=" * 60)
print("FINAL RAG ANSWER")
print("=" * 60)
print(answer)


# STEP 12: ASK MORE QUESTIONS
# Enter a question, or type exit to stop.

while True:
    user_question = input(
        "\nAsk another question (or type exit): "
    ).strip()

    if user_question.lower() in {"exit", "quit"}:
        print("RAG demonstration completed.")
        break

    if not user_question:
        print("Please enter a question.")
        continue

    try:
        response = rag_chain.invoke(user_question)
        print("\nANSWER:")
        print(response)

    except Exception as error:
        print("Could not generate the answer:", error)


STEP 1: Knowledge document created.
STEP 2: Documents loaded: 1
STEP 3: Number of chunks: 4

Chunk 1:
Artificial Intelligence (AI) is a field of computer science
that develops systems capable of performing intelligent tasks.

Machine Learning (ML) is a subset of AI. It enables computers
to learn patterns from data and make predictions.

Chunk 2:
Deep Learning is a subset of machine learning that uses
multi-layer neural networks.

Natural Language Processing (NLP) enables computers to
understand and generate human language.

Chunk 3:
Retrieval-Augmented Generation (RAG) combines document
retrieval with a language model to generate answers using
information retrieved from a knowledge base.

LangChain connects language models, prompts, retrievers,
document loaders and other components.

Chunk 4:
ChromaDB is a vector database that stores embeddings and
retrieves relevant text using vector similarity.

Ollama allows language models and embedding models to
run locally on a computer.

STEP 4:


Ask another question (or type exit):  exit


RAG demonstration completed.


In [8]:
%pip install -U langchain-chroma

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
Using cached langchain_chroma-1.1.0-py3-none-any.whl (12 kB)
Note: you may need to restart the kernel to use updated packages.
